
Task: Predict the next day’s minimum temperature based on previous 30 days.

Dataset: Daily minimum temperatures in Melbourne (CSV من GitHub)

Goal: Train 3 models: RNN, LSTM, GRU → compare training loss, validation loss, and predictions.

Bonus: Visualize predictions vs actual values.


In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"


In [ ]:
# 1️⃣ Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models


In [ ]:

# 2️⃣ Load Dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
df = pd.read_csv(url)
data = df['Temp'].values.astype(float).reshape(-1,1)

In [ ]:
# 3️⃣ Normalize Data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

In [ ]:
# 4️⃣ Create Sequences (Sliding Window)
seq_len = 30
X, y = [], []
for i in range(len(data_scaled) - seq_len):
    X.append(data_scaled[i:i+seq_len])
    y.append(data_scaled[i+seq_len])
X = np.array(X)
y = np.array(y)

In [ ]:
# 5️⃣ Train/Validation Split
split = int(len(X)*0.9)
X_train, y_train = X[:split], y[:split]
X_val, y_val = X[split:], y[split:]

In [ ]:
# 6️⃣ Function to build model
def build_model(model_type='RNN'):
    model = models.Sequential()
    if model_type=='RNN':
        model.add(layers.SimpleRNN(50, input_shape=(seq_len,1)))
    elif model_type=='LSTM':
        model.add(layers.LSTM(50, input_shape=(seq_len,1)))
    elif model_type=='GRU':
        model.add(layers.GRU(50, input_shape=(seq_len,1)))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# 7️⃣ Train Models
histories = {}
models_dict = {}
for m in ['RNN','LSTM','GRU']:
    print(f"Training {m}...")
    model = build_model(m)
    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=1
    )
    histories[m] = history
    models_dict[m] = model
    print(f"{m} training done.")

In [ ]:
# 8️⃣ Plot Loss Curves
plt.figure(figsize=(10,5))
for m in histories:
    plt.plot(histories[m].history['loss'], label=f'{m} Train')
    plt.plot(histories[m].history['val_loss'], label=f'{m} Val')
plt.title("Training & Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

In [ ]:
# 9️⃣ Predict on Validation Set
plt.figure(figsize=(12,5))
for m in models_dict:
    y_pred = models_dict[m].predict(X_val)
    y_pred_inv = scaler.inverse_transform(y_pred)
    y_val_inv = scaler.inverse_transform(y_val)
    plt.plot(y_val_inv, label='Real', color='black')
    plt.plot(y_pred_inv, label=f'{m} Pred')
    plt.title(f"{m} Predictions vs Real")
    plt.legend()
    plt.show()